PARTIE A — Chargement & choix des variables

In [1]:
# PARTIE A — Chargement & choix des variables

import pandas as pd, numpy as np 
import warnings; warnings.filterwarnings('ignore')

#df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf-8-sig') 
  
df = pd.read_excel('../data/dataset_assurance_ML.xlsx') 
df.to_csv('../data/dataset_assurance_ML.csv', index=False, encoding='utf-8-sig') 
df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf-8-sig')  
print(df.shape)

print('\nNombre de valeurs manquantes : \n',df.isnull().sum())
print('\nDoublons :',df.duplicated().sum())


(500, 27)

Nombre de valeurs manquantes : 
 N° Police                 0
Nom                       0
Prénom                    0
Sexe                      0
Âge                       0
Catégorie Prof.           0
Salaire Annuel (€)        0
Ville                     0
Code Postal               0
Type Contrat              0
Statut Contrat            0
Date Souscription         0
Date Échéance             0
Ancienneté (mois)         0
Prime Annuelle (€)        0
Franchise (€)             0
Nb Garanties              0
Type Véhicule             0
Usage Véhicule            0
Puissance Fiscale (CV)    0
Valeur Véhicule (€)       0
Coeff. Bonus-Malus        0
Nb Sinistres (3 ans)      0
Montant Sinistres (€)     0
Dernier Sinistre          0
Score Risque (0-100)      0
Résiliation               0
dtype: int64

Doublons : 0


In [2]:
#Etape 2 : la variable cible
TARGET = 'Résiliation' 
print(df[TARGET].value_counts()) 
print(df[TARGET].value_counts(normalize=True).round(2)) 

Résiliation
0    450
1     50
Name: count, dtype: int64
Résiliation
0    0.9
1    0.1
Name: proportion, dtype: float64


In [3]:
#Etape 3
print(pd.crosstab(df['Statut Contrat'], df[TARGET]))

Résiliation       0   1
Statut Contrat         
Actif           450   0
Résilié           0  36
Suspendu          0  14


In [4]:
#Etape 4
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)','Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)', 'Montant Sinistres (€)', 'Score Risque (0-100)'] 
cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre'] 
X = df[num_cols + cat_cols] 
y = df[TARGET] 
print(X.shape, y.shape)

(500, 12) (500,)


In [5]:
#Etape 5
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False)) 
print((df.groupby('Dernier Sinistre')[TARGET].mean()*100).round(2).sort_values()) 

Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412
Montant Sinistres (€)    0.284
Prime Annuelle (€)       0.041
Âge                     -0.006
Salaire Annuel (€)      -0.012
Ancienneté (mois)       -0.055
dtype: float64
Dernier Sinistre
Aucun                     2.65
Incendie                 15.79
Catastrophe naturelle    17.65
Accident                 26.09
Vol                      29.63
Bris de glace            30.00
Dégât des eaux           36.00
Name: Résiliation, dtype: float64


Les 3 variables numériques les plus liées à la résiliation sont:
Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412

PARTIE B — Pipeline, entraînement & évaluation

In [6]:
#PARTIE B — Pipeline, entraînement & évaluation

#Etape 6
from sklearn.model_selection import train_test_split 
  
X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size=0.2, random_state=42, stratify=y) 
print(X_train.shape, X_test.shape) 
print(y_train.mean().round(2), y_test.mean().round(2))

(400, 12) (100, 12)
0.1 0.1


In [7]:
#Etape 7
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
  
preprocessor = ColumnTransformer([ 
    ('num', StandardScaler(), num_cols), 
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols), 
])

In [8]:
#Etape 8
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier 
  
candidats = { 
    'Régression Logistique': LogisticRegression( 
        max_iter=1000, class_weight='balanced', random_state=42), 
    'Random Forest': RandomForestClassifier( 
        n_estimators=300, max_depth=4, min_samples_leaf=10, 
        class_weight='balanced', random_state=42), 
} 
pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)]) 
             for nom, algo in candidats.items()}

In [9]:
#Etape 9
from sklearn.model_selection import cross_val_score 

for nom, pipe in pipelines.items(): 
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc') 
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}') 

Régression Logistique  AUC = 0.745 ± 0.113
Random Forest          AUC = 0.801 ± 0.096


In [10]:
#Etape 10
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, 
confusion_matrix, classification_report) 

pipeline = pipelines['Random Forest'] 
pipeline.fit(X_train, y_train) 
y_pred  = pipeline.predict(X_test) 
y_proba = pipeline.predict_proba(X_test)[:, 1] 
print('Accuracy :', round(accuracy_score(y_test, y_pred),3))
print('F1       :', round(f1_score(y_test, y_pred), 3)) 
print('ROC-AUC  :', round(roc_auc_score(y_test, y_proba), 3)) 
print(confusion_matrix(y_test, y_pred)) 
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie'])) 

Accuracy : 0.87
F1       : 0.519
ROC-AUC  : 0.853
[[80 10]
 [ 3  7]]
              precision    recall  f1-score   support

       Reste       0.96      0.89      0.92        90
     Résilie       0.41      0.70      0.52        10

    accuracy                           0.87       100
   macro avg       0.69      0.79      0.72       100
weighted avg       0.91      0.87      0.88       100



#Etape 11

PARTIE C — Sauvegarde & interrogation du modèle

In [11]:
#PARTIE C — Sauvegarde & interrogation du modèle

#Etape 12
import joblib, os 

os.makedirs('../models', exist_ok=True)
joblib.dump(pipeline, '../models/pipeline_resiliation.pkl') 
print(os.path.getsize('../models/pipeline_resiliation.pkl') / 1024, 'Ko')

493.689453125 Ko


In [12]:
#Etape 13
import json 

meta = { 
    'modele': 'Random Forest', 
    'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3), 
    'num_cols': num_cols, 
    'cat_cols': cat_cols, 
    'num_ranges': {c: {'min': float(X[c].min()), 'max': float(X[c].max()),'median': float(X[c].median())} for c in num_cols}, 
    'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols}, 
} 
with open('../models/metadata.json', 'w', encoding='utf-8') as f: 
    json.dump(meta, f, ensure_ascii=False, indent=2) 

In [13]:
#Etape 14

modele = joblib.load('../models/pipeline_resiliation.pkl') 
  
client = pd.DataFrame([{ 
    'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950, 
    'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25, 'Nb Sinistres (3 ans)': 3, 
    'Montant Sinistres (€)': 4200, 'Score Risque (0-100)': 72, 
    'Type Contrat': 'Bronze', 'Catégorie Prof.': 'Entrepreneur', 
    'Usage Véhicule': 'Professionnel', 'Dernier Sinistre': 'Vol', 
}]) 
print('Classe :', modele.predict(client)) 
print('Proba  :', modele.predict_proba(client)[0, 1].round(3))

Classe : [1]
Proba  : 0.816


In [14]:
#Etape 15
try: 
    modele.predict(client.drop(columns=['Score Risque (0-100)'])) 
except Exception as e: 
    print('ERREUR :', e)


ERREUR : columns are missing: {'Score Risque (0-100)'}


PARTIE D — Interface Streamlit

In [15]:
#PARTIE D — Interface Streamlit

#Etape 16
